# GPT-2 124M Fine-tuning — Glaive Function Calling

Works on **Kaggle** and **Google Colab**.
- Kaggle: set Accelerator to **GPU T4 x2**, then **Save Version → Save & Run All**
- Colab: **Runtime → Change runtime type → T4 GPU**, then Run All


In [ ]:
!pip install -q tiktoken matplotlib tqdm pandas huggingface-hub datasets

import os, sys, torch

# Detect platform and set working directory
if os.path.exists('/kaggle/working'):
    WORKING_DIR = '/kaggle/working'
    PLATFORM = 'kaggle'
else:
    WORKING_DIR = '/content'
    PLATFORM = 'colab'

print(f'Platform   : {PLATFORM}')
print(f'Working dir: {WORKING_DIR}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device     : {device}')
if device == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected — enable accelerator before running.')


In [ ]:
%%writefile gpt_download.py
# Copyright (c) Sebastian Raschka under Apache License 2.0 (see LICENSE.txt).
# Source for "Build a Large Language Model From Scratch"
#   - https://www.manning.com/books/build-a-large-language-model-from-scratch
# Code: https://github.com/rasbt/LLMs-from-scratch


import os
import json

import numpy as np
import requests
import tensorflow as tf
from tqdm import tqdm


def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    backup_base_url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        backup_url = os.path.join(backup_base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        download_file(file_url, file_path, backup_url)

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params


def download_file(url, destination, backup_url=None):
    def _attempt_download(download_url):
        response = requests.get(download_url, stream=True, timeout=60)
        response.raise_for_status()

        file_size = int(response.headers.get("Content-Length", 0))

        # Check if file exists and has same size
        if os.path.exists(destination):
            file_size_local = os.path.getsize(destination)
            if file_size and file_size == file_size_local:
                print(f"File already exists and is up-to-date: {destination}")
                return True

        block_size = 1024  # 1 KB
        desc = os.path.basename(download_url)
        with tqdm(total=file_size, unit="iB", unit_scale=True, desc=desc) as progress_bar:
            with open(destination, "wb") as file:
                for chunk in response.iter_content(chunk_size=block_size):
                    if chunk:
                        file.write(chunk)
                        progress_bar.update(len(chunk))
        return True

    try:
        if _attempt_download(url):
            return
    except requests.exceptions.RequestException:
        if backup_url is not None:
            print(f"Primary URL ({url}) failed. Attempting backup URL: {backup_url}")
            try:
                if _attempt_download(backup_url):
                    return
            except requests.exceptions.RequestException:
                pass

        error_message = (
            f"Failed to download from both primary URL ({url})"
            f"{' and backup URL (' + backup_url + ')' if backup_url else ''}."
            "\nCheck your internet connection or the file availability.\n"
            "For help, visit: https://github.com/rasbt/LLMs-from-scratch/discussions/273"
        )
        print(error_message)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params

In [ ]:
%%writefile model.py
import math

import numpy as np
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    _coeff = math.sqrt(2.0 / math.pi)

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            self._coeff * (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_resid = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_resid(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_resid(x)
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))


def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [ ]:
%%writefile data.py
import torch
from torch.utils.data import Dataset


def format_entry(entry):
    system = entry['system'].strip().replace('SYSTEM:', '###SYSTEM:')

    turns = [
        t.strip()
         .replace("<|endoftext|>", "")
         .replace('USER:', '###USER:')
         .replace('ASSISTANT:', '###ASSISTANT:')
        for t in entry['chat'].split('\n\n\n')
        if t.strip()
    ]

    instruction_turns = []
    response = ""

    for turn in turns:
        if turn.startswith('###ASSISTANT:'):
            response = turn
            break
        instruction_turns.append(turn)

    instruction = system + '\n' + '\n'.join(instruction_turns)
    return instruction, response


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, allowed_max_length=1024):
        self.data = data
        self.encoded_texts = []
        self.allowed_max_length = allowed_max_length

        for entry in data:
            prompt, response = format_entry(entry)
            if not response:
                continue

            prompt_ids = tokenizer.encode(prompt)
            response_ids = tokenizer.encode(response)
            token_ids = prompt_ids + response_ids

            if len(token_ids) + 1 > self.allowed_max_length:
                continue

            self.encoded_texts.append((token_ids, len(prompt_ids)))

    def __len__(self):
        return len(self.encoded_texts)

    def __getitem__(self, index):
        return self.encoded_texts[index]


def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(token_ids) + 1 for token_ids, _ in batch)
    if allowed_max_length is not None:
        batch_max_length = min(batch_max_length, allowed_max_length)

    inputs_lst, targets_lst = [], []

    for token_ids, prompt_len in batch:
        new_item = token_ids.copy()
        new_item += [pad_token_id]  # EOS token
        if allowed_max_length is not None:
            new_item = new_item[:allowed_max_length]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # Mask padding tokens in targets (keep EOS, mask the rest)
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # Mask prompt tokens so loss is only computed on the response
        targets[:prompt_len - 1] = ignore_index

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [ ]:
%%writefile train.py
import argparse
import functools
import os
import time

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import pandas as pd
import tiktoken
import torch
from torch.utils.data import DataLoader

from data import format_entry, InstructionDataset, custom_collate_fn
from gpt_download import download_and_load_gpt2
from model import GPTModel, load_weights_into_gpt


# ---------------------------------------------------------------------------
# Training utilities
# ---------------------------------------------------------------------------

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    num_batches = min(num_batches, len(data_loader)) if num_batches else len(data_loader)
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
        else:
            break
    return total_loss / num_batches


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = torch.tensor(
        tokenizer.encode(start_context, allowed_special={"<|endoftext|>"})
    ).unsqueeze(0).to(device)
    with torch.no_grad():
        idx = encoded
        for _ in range(50):
            idx_cond = idx[:, -context_size:]
            logits = model(idx_cond)
            idx_next = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            if idx_next == 50256:
                break
            idx = torch.cat((idx, idx_next), dim=1)
        decoded = tokenizer.decode(idx.squeeze(0).tolist())
        print(decoded.replace("\n", " "))
    model.train()


def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses, output_path):
    fig, ax1 = plt.subplots(figsize=(5, 3))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.savefig(output_path)
    print(f"Loss plot saved to {output_path}")


# ---------------------------------------------------------------------------
# Model configs
# ---------------------------------------------------------------------------

MODEL_CONFIGS = {
    "124M":  {"emb_dim": 768,  "n_layers": 12, "n_heads": 12},
    "355M":  {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "774M":  {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "1558M": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
}


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Fine-tune GPT-2 on Glaive function calling")
    parser.add_argument("--model-size",         default="124M",    choices=list(MODEL_CONFIGS))
    parser.add_argument("--num-epochs",         type=int,   default=2)
    parser.add_argument("--batch-size",         type=int,   default=2)
    parser.add_argument("--allowed-max-length", type=int,   default=512)
    parser.add_argument("--lr",                 type=float, default=5e-5)
    parser.add_argument("--weight-decay",       type=float, default=0.1)
    parser.add_argument("--data-slice",         type=int,   default=1000,
                        help="Number of dataset samples to use (use -1 for full dataset)")
    parser.add_argument("--eval-freq",          type=int,   default=5)
    parser.add_argument("--eval-iter",          type=int,   default=5)
    parser.add_argument("--models-dir",         default="notebooks/gpt2",
                        help="Directory to store/load GPT-2 TF checkpoints")
    parser.add_argument("--output-dir",         default="checkpoints")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    # --- Device ---
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Device: {device}")

    # --- Dataset ---
    print("Loading Glaive function calling dataset...")
    df = pd.read_json("hf://datasets/glaiveai/glaive-function-calling-v2/glaive-function-calling-v2.json")
    data = df.to_dict("records")
    if args.data_slice > 0:
        data = data[:args.data_slice]
    print(f"Loaded {len(data)} samples")

    train_portion = int(len(data) * 0.85)
    val_portion   = int(len(data) * 0.10)
    train_data = data[:train_portion]
    val_data   = data[train_portion:train_portion + val_portion]
    test_data  = data[train_portion + val_portion:]
    print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

    tokenizer = tiktoken.get_encoding("gpt2")
    collate_fn = functools.partial(custom_collate_fn, allowed_max_length=args.allowed_max_length)

    train_loader = DataLoader(
        InstructionDataset(train_data, tokenizer, allowed_max_length=args.allowed_max_length),
        batch_size=args.batch_size, collate_fn=collate_fn, shuffle=True, drop_last=False,
    )
    val_loader = DataLoader(
        InstructionDataset(val_data, tokenizer, allowed_max_length=args.allowed_max_length),
        batch_size=args.batch_size, collate_fn=collate_fn, shuffle=False, drop_last=False,
    )
    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    # --- Model ---
    cfg = {**BASE_CONFIG, **MODEL_CONFIGS[args.model_size]}
    print(f"Downloading/loading GPT-2 {args.model_size} weights...")
    _, params = download_and_load_gpt2(model_size=args.model_size, models_dir=args.models_dir)
    model = GPTModel(cfg)
    load_weights_into_gpt(model, params)
    model.to(device)
    print(f"GPT-2 {args.model_size} loaded and weights transferred")

    # --- Baseline loss ---
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
        val_loss   = calc_loss_loader(val_loader,   model, device, num_batches=5)
    print(f"Baseline — Train loss: {train_loss:.3f} | Val loss: {val_loss:.3f}")

    # --- Training ---
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    start_context = format_entry(val_data[0])[0]

    start_time = time.time()
    train_losses, val_losses, tokens_seen = train_model_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=args.num_epochs,
        eval_freq=args.eval_freq,
        eval_iter=args.eval_iter,
        start_context=start_context,
        tokenizer=tokenizer,
    )
    elapsed = (time.time() - start_time) / 60
    print(f"Training completed in {elapsed:.2f} minutes")

    # --- Save checkpoint ---
    ckpt_path = os.path.join(args.output_dir, f"gpt2-{args.model_size}-function-calling.pth")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Checkpoint saved to {ckpt_path}")

    # --- Loss plot ---
    if train_losses:
        epochs_seen = torch.linspace(0, args.num_epochs, len(train_losses))
        plot_losses(epochs_seen, tokens_seen, train_losses, val_losses,
                    output_path=os.path.join(args.output_dir, "loss-plot.pdf"))


if __name__ == "__main__":
    main()

## Config
Edit these values, then run the Train cell.


In [ ]:
MODEL_SIZE      = '124M'  # 124M / 355M / 774M / 1558M
NUM_EPOCHS      = 2
BATCH_SIZE      = 8
ALLOWED_MAX_LEN = 512
DATA_SLICE      = 100     # smoke test — change to -1 for full dataset
MODELS_DIR      = os.path.join(WORKING_DIR, 'gpt2')
OUTPUT_DIR      = os.path.join(WORKING_DIR, 'checkpoints')


## (Optional — Colab only) Save checkpoint to Google Drive
Skip on Kaggle — outputs are saved automatically.


In [ ]:
# if PLATFORM == 'colab':
#     from google.colab import drive
#     drive.mount('/content/drive')
#     OUTPUT_DIR = '/content/drive/MyDrive/gpt2-finetune/checkpoints'
#     os.makedirs(OUTPUT_DIR, exist_ok=True)


## Train


In [ ]:
import runpy, sys, os

# Force single GPU to prevent double-run on T4 x2
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_py = os.path.join(WORKING_DIR, 'train.py')

sys.argv = [
    'train.py',
    '--model-size',         MODEL_SIZE,
    '--num-epochs',         str(NUM_EPOCHS),
    '--batch-size',         str(BATCH_SIZE),
    '--allowed-max-length', str(ALLOWED_MAX_LEN),
    '--data-slice',         str(DATA_SLICE),
    '--models-dir',         MODELS_DIR,
    '--output-dir',         OUTPUT_DIR,
]

runpy.run_path(train_py, run_name='__main__')


## Checkpoint


In [ ]:
ckpt = os.path.join(OUTPUT_DIR, f'gpt2-{MODEL_SIZE}-function-calling.pth')
print('Checkpoint:', ckpt)
print('Exists    :', os.path.exists(ckpt))

if PLATFORM == 'colab' and os.path.exists(ckpt):
    from google.colab import files
    files.download(ckpt)

if PLATFORM == 'kaggle':
    print('On Kaggle: download from the Output tab (right panel).')
